# Agentic Architecture 4: Planning

In this notebook, we explore the **Planning** architecture. This pattern introduces a crucial layer of foresight into an agent's reasoning process. Instead of reacting to information step-by-step as in the ReAct model, a planning agent first decomposes a complex task into a sequence of smaller, manageable sub-goals. It creates a full 'battle plan' before taking any action.

This proactive approach brings structure, predictability, and efficiency to multi-step tasks. To highlight its benefits, we will directly compare the performance of a **reactive agent (ReAct)** against our new **planning agent**. We will present both with a task that requires gathering multiple pieces of information before performing a final calculation, demonstrating how a pre-computed plan can lead to a more robust and direct solution.

### Definition

The **Planning** architecture involves an agent that explicitly breaks down a complex goal into a detailed sequence of sub-tasks before beginning execution. The output of this initial planning phase is a concrete, step-by-step plan that the agent then follows methodically to reach the solution.

### High-level Workflow

1. **Receive Goal:** The agent is given a complex task.
2. **Plan:** A dedicated 'Planner' component analyzes the goal and generates an ordered list of sub-tasks required to achieve it. For example: `["Find fact A", "Find fact B", "Calculate C using A and B"]`.
3. **Execute:** An 'Executor' component takes the plan and carries out each sub-task in sequence, using tools as needed.
4. **Synthesize:** Once all steps in the plan are complete, a final component synthesizes the results from the executed steps into a coherent final answer.

### When to Use / Applications

- **Multi-Step Workflows:** Ideal for tasks where the sequence of operations is known and critical, such as generating a report that requires fetching data, processing it, and then summarizing it.
- **Project Management Assistants:** Decomposing a large goal like "launch a new feature" into sub-tasks for different teams.
- **Educational Tutoring:** Creating a lesson plan to teach a student a specific concept, from foundational principles to advanced application.

### Strengths & Weaknesses

- **Strengths:**
    - **Structured & Traceable:** The entire workflow is laid out in advance, making the agent's process transparent and easy to debug.
    - **Efficient:** Can be more efficient than ReAct for predictable tasks, as it avoids unnecessary reasoning loops between steps.
- **Weaknesses:**
    - **Brittle to Change:** A pre-made plan can fail if the environment changes unexpectedly during execution. It's less adaptive than a ReAct agent, which can change its mind after every step.


## Phase 0: Foundation & Setup

We'll begin with our standard setup process: installing libraries and configuring API keys for OpenRouter, LangSmith, and our Tavily web search tool.

In [ ]:
import os
import re
from typing import Annotated, TypedDict, Optional
from dotenv import load_dotenv

# Pydantic for data modeling / validation
from pydantic import BaseModel, Field

# LangChain components
from langchain_openrouter import ChatOpenRouter                    # OpenRouter LLM wrapper
from langchain_tavily import TavilySearch                          # Tavily search tool wrapper 
from langchain_core.messages import ToolMessage, SystemMessage     # Base class for messages in LangChain
from langchain_core.tools import tool                              # Decorator for defining tools in LangChain

# LangGraph components
from langgraph.graph import StateGraph, END                        # Build a state machine graph
from langgraph.graph.message import AnyMessage, add_messages       # For handling messages in the graph
from langgraph.prebuilt import ToolNode, tools_condition           # Prebuilt nodes and conditions

# For pretty printing 
from rich.console import Console  
from rich.markdown import Markdown     


# --- API Key and Tracing Setup ---
load_dotenv()  

# Enable LangSmith tracing for monitoring and debugging
os.environ["LANGSMITH_PROJECT"] = "Agentic Architecture - Planning (OpenRouter)"

# Verify that all required API keys are available
for key in ["OPENROUTER_API_KEY", "LANGSMITH_API_KEY", "TAVILY_API_KEY"]:
    if not os.environ.get(key):
        print(f"{key} not found. Please create a .env file and set it.")

print("Environment variables loaded and everything is set up.")

Environment variables loaded and everything is set up.


## Phase 1: The Baseline - A Reactive Agent (ReAct)

To appreciate the value of planning, we first need a baseline. We will use the ReAct agent we built in the previous notebook. This agent is intelligent but myopic—it figures out its path one step at a time.

### Step 1.1: Re-building the ReAct Agent

We will quickly reconstruct the ReAct agent. Its core feature is a loop where the agent's output is routed back to itself after every tool call, allowing it to reassess and decide its next move based on the latest information.

In [2]:
# Define the LLM 
llm = ChatOpenRouter(
    model="openrouter/free", 
    temperature=0
)

console = Console()

In [3]:
# Define the state for our graphs
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

# 1. Define the base tool from the tavily package
tavily_search_tool = TavilySearch(max_results=2)

# 2. Custom Tool: The .invoke() method already returns a clean string, so we just pass it through.
@tool
def web_search(query: str) -> str:
    """Performs a web search using Tavily and returns the results as a string."""
    console.print(f"--- TOOL: Searching for '{query}'...")
    results = tavily_search_tool.invoke(query)
    return results

# 3. Bind the LLM to our custom tool
llm_with_tools = llm.bind_tools([web_search])

# 4. Agent node with a system prompt to force one tool call at a time
def react_agent_node(state: AgentState):
    console.print("--- REACTIVE AGENT: Thinking... ---")
    
    messages_with_system_prompt = [
        SystemMessage(content="You are a helpful research assistant. You must call one and only one tool at a time. Do not call multiple tools in a single turn. After receiving the result from a tool, you will decide on the next step.")
    ] + state["messages"]

    response = llm_with_tools.invoke(messages_with_system_prompt)
    
    return {"messages": [response]}

# 5. Use our custom tool in the ToolNode
tool_node = ToolNode([web_search])

# The ReAct graph with its characteristic loop
react_graph_builder = StateGraph(AgentState)
react_graph_builder.add_node("agent", react_agent_node)
react_graph_builder.add_node("tools", tool_node)

react_graph_builder.set_entry_point("agent")
react_graph_builder.add_conditional_edges("agent", tools_condition)
react_graph_builder.add_edge("tools", "agent")

react_agent_app = react_graph_builder.compile()
print("Reactive (ReAct) agent compiled successfully.")

Reactive (ReAct) agent compiled successfully.


### Step 1.2: Testing the Reactive Agent on a Plan-Centric Problem

We will give the ReAct agent a task that requires two distinct data-gathering steps followed by a final calculation. This will test its ability to manage a multi-step workflow without an upfront plan.

In [4]:
plan_centric_query = """
Find the population of the capital cities of France, Germany, and Italy. 
Then calculate their combined total. 
Finally, compare that combined total to the population of the United States, and say which is larger.
"""

console.print(f"[bold yellow]Testing REACTIVE agent on a plan-centric query:[/bold yellow] '{plan_centric_query}'\n")

final_react_output = None
for chunk in react_agent_app.stream({"messages": [("user", plan_centric_query)]}, stream_mode="values"):
    final_react_output = chunk
    console.print(f"--- [bold purple]Current State Update[/bold purple] ---")
    chunk['messages'][-1].pretty_print()
    console.print("\n")
    
console.print("\n--- [bold red]Final Output from Reactive Agent[/bold red] ---")
console.print(Markdown(final_react_output['messages'][-1].content))

Testing REACTIVE agent on a plan-centric query: '
Find the population of the capital cities of France, Germany, and Italy. 
Then calculate their combined total. 
Finally, compare that combined total to the population of the United States, and say which is larger.
'

--- Current State Update ---

================================ Human Message =================================


Find the population of the capital cities of France, Germany, and Italy. 
Then calculate their combined total. 
Finally, compare that combined total to the population of the United States, and say which is larger.



--- REACTIVE AGENT: Thinking... ---

--- Current State Update ---

...

--- REACTIVE AGENT: Thinking... ---

--- Current State Update ---

================================== Ai Message ==================================

**Capital city populations (2024‑2026 estimates)**  

| Country | Capital | Population (city limits) |
|---------|---------|--------------------------|
| France  | Paris   | **≈ 2,048,472** |
| Germany | Berlin  | **≈ 3,662,381** |
| Italy   | Rome    | **≈ 4,361,370** |

**Combined total**

2,048,472 (Paris)  
+ 3,662,381 (Berlin)  
+ 4,361,370 (Rome)  
= **10,072,223** residents in total.

**Comparison with the United States**

- U.S. population (July 1 2024 estimate): **≈ 340 million** (≈ 340,000,000).

**Result**

The combined population of Paris, Berlin, and Rome (**≈ 10 million**) is far smaller than the population of the United States (**≈ 340 million**). Therefore, the United States has a larger population.


--- Final Output from Reactive Agent ---

Capital city populations (2024‑2026 estimates)                                                                     

                                            
 Country  Capital  Population (city limits) 
 ────────────────────────────────────────── 
 France   Paris    ≈ 2,048,472              
 Germany  Berlin   ≈ 3,662,381              
 Italy    Rome     ≈ 4,361,370              
                                            

Combined total                                                                                                     

2,048,472 (Paris)                                                                                                  

 • 3,662,381 (Berlin)                                                                                              
 • 4,361,370 (Rome)                                                                                                
   = 10,072,223 residents in total.                                                                                

Comparison with the United States                                                                                  

 • U.S. population (July 1 2024 estimate): ≈ 340 million (≈ 340,000,000).                                          

Result                                                                                                             

The combined population of Paris, Berlin, and Rome (≈ 10 million) is far smaller than the population of the United 
States (≈ 340 million). Therefore, the United States has a larger population.

**Discussion of the Output:**

The ReAct agent successfully completed the task. By observing the streamed output, we can trace its step-by-step reasoning process:

1. It first decides to search for the population of all three cities together.
2. Then it tries to go one by one while incorporating the tool output into it's memory.
3. Finally, with all pieces of information gathered, it performs the calculation and provides the final answer.

While it works, this iterative discovery process is not always the most efficient. For a predictable task like this, the agent is making extra LLM calls to reason between each step. This sets the stage for demonstrating the value of a planning agent.


## Phase 2: The Advanced Approach - A Planning Agent

Now, let's build an agent that thinks before it acts. This agent will have a dedicated Planner to create a complete task list, an Executor to carry out the plan, and a Synthesizer to assemble the final result.

### Step 2.1: Defining the Planner, Executor, and Synthesizer Nodes

We will create the core components for our new agent:

1. **Planner:** An LLM-based node that takes the user request and outputs a structured plan.
2. **Executor:** A node that takes the plan, executes the next step using a tool, and records the result.
3. **Synthesizer:** A final LLM-based node that takes all the collected results and generates the final answer.


In [ ]:
# Pydantic model to ensure the planner's output is a structured list of steps
class Plan(BaseModel):
    """A plan of tool calls to execute to answer the user's query."""
    steps: list[str] = Field(description="A list of tool calls that, when executed, will answer the query.")

# Define the state for the planning agent
class PlanningState(TypedDict):
    user_request: str
    plan: Optional[list[str]]
    intermediate_steps: list[ToolMessage]
    final_answer: Optional[str]

In [8]:
# Planner node that generates a plan of action
def planner_node(state: PlanningState):
    """Generates a plan of action to answer the user's request."""
    console.print("--- PLANNER: Decomposing task... ---")

    # An explicit prompt with a clear example (few-shot prompting)
    prompt = f"""You are an expert planner. Your job is to create a step-by-step plan to answer the user's request.
        Each step in the plan must be a single call to the `web_search` tool.

        **Instructions:**
        1. Analyze the user's request.
        2. Break it down into a sequence of simple, logical search queries.
        3. Format the output as a list of strings, where each string is a single valid tool call.

        **Example:**
        Request: "What is the capital of France and what is its population?"
        Correct Plan Output:
        [
            "web_search('capital of France')",
            "web_search('population of Paris')"
        ]

        **User's Request:**
        {state['user_request']}
    """

    planner_llm = llm.with_structured_output(Plan)
    plan_result = planner_llm.invoke(prompt)

    console.print(f"--- PLANNER: Generated Plan: {plan_result.steps} ---")
    return {"plan": plan_result.steps}

print("Planner node defined.")

Planner node defined.


In [ ]:
# Executor node that runs the generated plan
def executor_node(state: PlanningState):
    """Executes the next step in the plan."""
    console.print("--- EXECUTOR: Running next step... ---")

    plan = state["plan"]
    next_step = plan[0]

    # Robust regex to handle both single and double quotes
    match = re.search(r"(\w+)\((?:\"|\')(.*?)(?:\"|\')\)", next_step)
    if not match:
        tool_name = "web_search"
        query = next_step
    else:
        tool_name, query = match.groups()[0], match.groups()[1]
    
    console.print(f"--- EXECUTOR: Calling tool '{tool_name}' with query '{query}' ---")
    
    result = tavily_search_tool.invoke(query)
    
    # We still create a ToolMessage, but the tool call itself is now safe.
    tool_message = ToolMessage(
        content=str(result),
        name=tool_name,
        tool_call_id=f"manual-{hash(query)}"
    )
    
    return {
        "plan": plan[1:], # Pop the executed step from the plan
        "intermediate_steps": state["intermediate_steps"] + [tool_message]
    }

print("Executor node defined.")

Executor node defined.


In [10]:
# Synthesizer node that generates the final answer from intermediate steps
def synthesizer_node(state: PlanningState):
    """Synthesizes the final answer from the intermediate steps."""
    console.print("--- SYNTHESIZER: Generating final answer... ---")
    
    context = "\n".join([f"Tool {msg.name} returned: {msg.content}" for msg in state["intermediate_steps"]])
    
    prompt = f"""You are an expert synthesizer. Based on the user's request and the collected data, provide a comprehensive final answer.
        Request: {state['user_request']}
        Collected Data:
        {context}
    """
    
    final_answer = llm.invoke(prompt).content
    return {"final_answer": final_answer}

print("Synthesizer node defined.")

Synthesizer node defined.


### Step 2.2: Building the Planning Agent Graph

Now we will assemble the new nodes into a graph. The flow will be: `Planner -> Executor (looped) -> Synthesizer`.

In [12]:
# Router to decide whether to continue executing the plan or move to synthesis
def planning_router(state: PlanningState):
    if not state["plan"]:
        console.print("--- ROUTER: Plan complete. Moving to synthesizer. ---")
        return "synthesize"
    else:
        console.print("--- ROUTER: Plan has more steps. Continuing execution. ---")
        return "execute"

In [13]:
# Build the planning graph
planning_graph_builder = StateGraph(PlanningState)

planning_graph_builder.add_node("plan", planner_node)
planning_graph_builder.add_node("execute", executor_node)
planning_graph_builder.add_node("synthesize", synthesizer_node)

planning_graph_builder.set_entry_point("plan")
# Route after planning
planning_graph_builder.add_conditional_edges(
    "plan", 
    planning_router, 
    {
        "execute": "execute", 
        "synthesize": "synthesize"
    }
) 
# Route after execution
planning_graph_builder.add_conditional_edges(
    "execute", 
    planning_router, 
    {
        "execute": "execute", 
        "synthesize": "synthesize"
    }
)
planning_graph_builder.add_edge("synthesize", END)

planning_agent_app = planning_graph_builder.compile()
print("Planning agent compiled successfully.")

Planning agent compiled successfully.


## Phase 3: Head-to-Head Comparison

Let's run our new planning agent on the same task and compare its execution flow and final output to the reactive agent.


In [14]:
console.print(f"[bold green]Testing PLANNING agent on the same plan-centric query:[/bold green] '{plan_centric_query}'\n")

# Remember to initialize the state correctly, especially the list for intermediate steps
initial_planning_input = {
    "user_request": plan_centric_query, 
    "intermediate_steps": []
}

final_planning_output = planning_agent_app.invoke(initial_planning_input)

console.print("\n--- [bold green]Final Output from Planning Agent[/bold green] ---")
console.print(Markdown(final_planning_output['final_answer']))

Testing PLANNING agent on the same plan-centric query: '
Find the population of the capital cities of France, Germany, and Italy. 
Then calculate their combined total. 
Finally, compare that combined total to the population of the United States, and say which is larger.
'

--- PLANNER: Decomposing task... ---

--- PLANNER: Generated Plan: ["web_search('population of Paris France')", "web_search('population of Berlin 
Germany')", "web_search('population of Rome Italy')", "web_search('population of United States')"] ---

--- ROUTER: Plan has more steps. Continuing execution. ---

--- EXECUTOR: Running next step... ---

...

--- EXECUTOR: Calling tool 'web_search' with query 'population of United States' ---

--- ROUTER: Plan complete. Moving to synthesizer. ---

--- SYNTHESIZER: Generating final answer... ---

--- Final Output from Planning Agent ---

Based on the collected data, here is a comprehensive synthesis of the population information and comparison:       

1. Population of Capital Cities                                                                                    

 • Paris (France):                                                                                                 
   The most recent estimate from World Population Review (2026) is 2,206,488.                                      
   Source: ]8;id=7730121;https://worldpopulationreview.com/cities/france/paris\World Population Review]8;;\.                                                                                
 • Berlin (Germany):                                                                                               
   The 2026 estimate from World Population Review is 3,775,697.                                                    
   Source: ]8;id=7730122;https://worldpopulationreview.com/cities/germany/berlin\World Population Review]8;;\.                                                                                
 • Rome (Italy):                                                                                                   
   The 2026 estimate from World Population Review is 4,361,370.                                                    
   Source: ]8;id=7730123;https://worldpopulationreview.com/cities/italy/rome\World Population Review]8;;\.                                                                                

2. Combined Total of Capital Cities                                                                                

 • Calculation:                                                                                                    
   Paris + Berlin + Rome = 2,206,488 + 3,775,697 + 4,361,370 = 10,343,555.                                         
 • Result: The combined population of Paris, Berlin, and Rome is 10.34 million.                                    

3. Comparison to the United States Population                                                                      

 • United States Population (2026):                                                                                
   Estimated at 341 million (based on live data from Worldometer and historical trends from Macrotrends).          
   Source: ]8;id=7730127;https://www.worldometers.info/world-population/us-population\Worldometer]8;;\.                                                                                            
 • Comparison:                                                                                                     
    • Combined capital cities: 10.34 million                                                                       
    • United States: 341 million                                                                                   
    • The United States is significantly larger, with a population ~33 times greater than the combined capitals of 
      France, Germany, and Italy.                                                                                  

Key Notes:                                                                                                         

 • Data Consistency: All capital city figures are from 2026 estimates (World Population Review), ensuring temporal 
   alignment.                                                                                                      
 • US Population: The exact live figure (e.g., 341,313,642 as of July 2026 per Worldometer) is used for accuracy.  
 • Context: The combined population of these three European capitals represents only ~3% of the U.S. population,   
   highlighting the vast scale of the U.S. compared to major European cities.                                      

Final Answer:                                                                                                      
The combined population of Paris, Berlin, and Rome is 10.34 million. The United 

**Discussion of the Output:**

The difference in process is immediately clear. The very first step was the `Planner` creating a complete, explicit plan: `['web_search("population of Paris")', 'web_search("population of Berlin")']`.

The agent then executed this plan methodically without needing to stop and think between steps. This process is:

- **More Transparent:** We can see the agent's entire strategy before it even starts.
- **More Robust:** It's less likely to get sidetracked because it's following a clear set of instructions.
- **Potentially More Efficient:** It avoids extra LLM calls for reasoning between steps.

This demonstrates the power of planning for tasks where the required steps can be determined in advance.

## Phase 4: Quantitative Evaluation

To formalize our comparison, we will use an LLM-as-a-Judge to score both agents, focusing on the quality and efficiency of their problem-solving process.

In [16]:
# Pydantic model for structured evaluation of the agent's process
class ProcessEvaluation(BaseModel):
    """Schema for evaluating an agent's problem-solving process."""
    task_completion_score: int = Field(description="Score 1-10 on whether the agent successfully completed the task.")
    process_efficiency_score: int = Field(description="Score 1-10 on the efficiency and directness of the agent's process. A higher score means a more logical and less roundabout path.")
    justification: str = Field(description="A brief justification for the scores.")

judge_llm = llm.with_structured_output(ProcessEvaluation)

# Function to evaluate the agent's process using the judge LLM
def evaluate_agent_process(query: str, final_state: dict):
    # For the ReAct agent, the trace is in 'messages'. For Planning, it's in 'intermediate_steps'.
    if 'messages' in final_state:
        trace = "\n".join([f"{m.type}: {str(m.content)}" for m in final_state['messages']])
    else:
        trace = f"Plan: {final_state.get('plan', [])}\nSteps: {final_state.get('intermediate_steps', [])}"
        
    prompt = f"""You are an expert judge of AI agents. Evaluate the agent's process for solving the task on a scale of 1-10.
    Focus on whether the process was logical and efficient.
    
    **User's Task:** {query}
    **Full Agent Trace:**\n```\n{trace}\n```
    """
    return judge_llm.invoke(prompt)

In [17]:
console.print("--- Evaluating Reactive Agent's Process ---")
react_agent_evaluation = evaluate_agent_process(plan_centric_query, final_react_output)
console.print(react_agent_evaluation.model_dump())

console.print("\n--- Evaluating Planning Agent's Process ---")
planning_agent_evaluation = evaluate_agent_process(plan_centric_query, final_planning_output)
console.print(planning_agent_evaluation.model_dump())

--- Evaluating Reactive Agent's Process ---

{
    'task_completion_score': 9,
    'process_efficiency_score': 7,
    'justification': 'The agent successfully gathered the required population data, calculated the combined total, 
and compared it to the U.S. population, providing a clear answer. However, the process involved multiple separate 
tool calls (a combined query that missed Rome, then individual queries for Rome and Berlin, plus a U.S. query) that
could have been streamlined into fewer, more targeted searches, reducing redundancy and improving efficiency.'
}

--- Evaluating Planning Agent's Process ---

{
    'task_completion_score': 10,
    'process_efficiency_score': 10,
    'justification': "The agent executed a perfectly logical and efficient process: it identified the required data
points (populations of Paris, Berlin, Rome, and the United States), retrieved each piece of information directly 
via targeted searches, summed the capital cities' populations, and compared the total to the U.S. population. The 
approach was straightforward, required no backtracking or clarification, and achieved the goal in the minimum 
number of steps."
}

**Discussion of the Output:** 

The judge's scores quantify the difference in the two approaches. Both agents likely receive a high `task_completion_score` as they both eventually find the answer. However, the `Planning Agent` received a significantly higher `process_efficiency_score`. The judge's justification highlights that its upfront plan was a more direct and logical way to solve the problem compared to the ReAct agent's step-by-step, exploratory process.

*This evaluation confirms our hypothesis:* for problems where the solution path is predictable, the Planning architecture offers a more structured, transparent, and efficient approach.

## Conclusion

In this notebook, we have implemented the **Planning** architecture and contrasted it directly with the **ReAct** pattern. By forcing an agent to first construct a comprehensive plan before execution, we gain significant benefits in transparency, robustness, and efficiency for well-defined, multi-step tasks.

While ReAct excels in exploratory scenarios where the next step is unknown, Planning shines when the path to a solution can be charted in advance. Understanding this trade-off is crucial for a system designer. Choosing the right architecture for the right problem is a key skill in building effective and intelligent AI agents. The Planning pattern is an essential tool in that toolkit, providing the structure needed for complex, predictable workflows.